# 🧬 Protein-Ligand 3D Complex Visualization
### Interactive Structural Analysis of Docked Complexes (AlphaFold 3 / TheraFlow)

This notebook provides an interactive 3D visualization dashboard for inspecting molecular docking poses and protein-ligand binding conformations predicted by **AlphaFold 3** or generated via **TheraFlow Docking / MD Runs**.

---

### 🌟 Key Features
- **Interactive 3D Viewport**: Rotate, pan, and zoom 3D structures directly within the notebook using `py3Dmol`. Supports both CIF (`.cif`) and PDB (`.pdb`) structures.
- **Rendering Modes**:
  - **Standard Cartoon**: Full protein backbone with spectrum rainbow coloring, binding pocket residues within 4.0 Å highlighted in cyan, and ligand in magenta sticks.
  - **Lock & Key Surface**: Solvent-excluded surface (SES) representation of the protein pocket illustrating steric complementarity.
  - **Super Clean**: Minimal grayscale cartoon focusing attention directly on the ligand binding pose.
- **Dynamic File Discovery**: Automatic `if/else` discovery checking for AlphaFold 3 CIF outputs or falling back to TheraFlow MD docking runs.
- **Snapshot Export**: Quick-capture button to download the current 3D canvas viewport as a high-resolution PNG image.


## 1. Environment & Library Setup
Import required dependencies for file handling, data processing, styling, and interactive Jupyter widgets.


In [1]:
import os
import re
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Javascript, clear_output

# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)

# Setup plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

## 2. 3D Molecular Rendering Functions
Define helper functions using `py3Dmol` to detect the ligand residue ID from CIF/PDB files and render customized 3D complex views.


In [2]:
try:
    import py3Dmol
except ImportError:
    print("py3Dmol not found. Installing...")
    %pip install -q py3Dmol
    import py3Dmol

def detect_ligand_resn(file_path):
    """Scan CIF or PDB file for ligand residue name (e.g., LIG_B, LIG, UNK, MOL)"""
    try:
        with open(file_path, 'r') as f:
            if file_path.endswith(('.cif', '.mmcif')):
                for line in f:
                    if 'LIG' in line:
                        match = re.search(r'\b(LIG_[A-Z0-9]+|LIG)\b', line)
                        if match:
                            return match.group(1)
            elif file_path.endswith('.pdb'):
                for line in f:
                    if line.startswith(('HETATM', 'ATOM')):
                        resname = line[17:20].strip()
                        if resname in ['LIG', 'UNK', 'MOL', 'DRG'] or resname.startswith('LIG'):
                            return resname
    except Exception:
        pass
    return "LIG" if file_path.endswith('.pdb') else "LIG_B"

def _get_format(file_path):
    return "cif" if file_path.endswith(('.cif', '.mmcif')) else "pdb"

def view_docked_pocket(file_path, ligand_resn):
    """Standard Cartoon: Rainbow protein cartoon + cyan pocket residues (<4.0A) + magenta ligand."""
    with open(file_path, 'r') as f:
        file_data = f.read()
    fmt = _get_format(file_path)
    viewer = py3Dmol.view(width=800, height=600)
    viewer.addModel(file_data, fmt)
    viewer.setStyle({'cartoon': {'color': 'spectrum'}})
    viewer.setStyle({'resn': ligand_resn}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.35}})
    viewer.addStyle({'within': {'distance': 4.0, 'byres': True, 'ref': {'resn': ligand_resn}}},
                    {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})
    viewer.addStyle({'resn': ligand_resn}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.35}})
    viewer.zoomTo({'resn': ligand_resn})
    viewer.setBackgroundColor('white')
    return viewer

def view_lock_and_key(file_path, ligand_resn):
    """Lock & Key Surface: Protein solvent-excluded surface + magenta ligand."""
    with open(file_path, 'r') as f:
        file_data = f.read()
    fmt = _get_format(file_path)
    viewer = py3Dmol.view(width=800, height=600)
    viewer.addModel(file_data, fmt)
    viewer.setStyle({'resn': ligand_resn}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.35}})
    viewer.addStyle({'within': {'distance': 4.0, 'byres': True, 'ref': {'resn': ligand_resn}}},
                    {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})
    viewer.addStyle({'resn': ligand_resn}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.35}})
    viewer.addSurface(py3Dmol.SES, {
        'opacity': 0.55, 
        'color': '#e0e0e0'
    }, {'not': {'resn': ligand_resn}})
    viewer.zoomTo({'resn': ligand_resn})
    viewer.setBackgroundColor('white')
    return viewer

def view_super_clean(file_path, ligand_resn):
    """Super Clean: Minimal grayscale backbone + magenta ligand."""
    with open(file_path, 'r') as f:
        file_data = f.read()
    fmt = _get_format(file_path)
    viewer = py3Dmol.view(width=800, height=600)
    viewer.addModel(file_data, fmt)
    viewer.setStyle({'cartoon': {'color': '#d3d3d3'}})
    viewer.setStyle({'resn': ligand_resn}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.35}})
    viewer.zoomTo({'resn': ligand_resn})
    viewer.setBackgroundColor('white')
    return viewer


## 3. Interactive Docking Viewer & Snapshot Export
Scan structure files with 1st priority given to **TheraFlow MD/docking runs** (`TheraFlow/outputs/docking_md_runs`), falling back to **AlphaFold 3** predictions (`alphafold3/output_result5`), filtered by `target_prefixes`.


In [3]:
# ==============================================================================
# 1. Structure Discovery (1st Priority: TheraFlow Docking/MD Runs, 2nd: AlphaFold 3)
# ==============================================================================

# Target prefixes to filter (leave empty [] to include all discovered models)
target_prefixes = [
    "docking_BiCEV_cardiomyopathy_50",
#     "docking_FAME_glaucoma_bulk_119",
    "docking_gexmolgen_glaucoma_283",
    "docking_SmilesGEN_VAE_parkinsons_248",
    "docking_BiAAE_hcc_30"
]

# Location 1: TheraFlow MD & Docking run structure files (1st priority)
theraflow_files = (
    glob.glob("TheraFlow/outputs/docking_md_runs/*/*/*.pdb") or
    glob.glob("outputs/docking_md_runs/*/*/*.pdb") or
    glob.glob("TheraFlow/outputs/docking_md_runs/*/*.cif") or
    glob.glob("outputs/docking_md_runs/*/*.cif")
)

# Location 2: AlphaFold 3 CIF files (2nd priority)
af3_files = (
    glob.glob("alphafold3/output_result5/*/*_model.cif") or 
    glob.glob("../alphafold3/output_result5/*/*_model.cif")
)

# If / Else Discovery Setup:
if theraflow_files:
    # Filter by target_prefixes if defined
    if target_prefixes:
        # Flexible match on filename or parent run directory (case-insensitive)
        filtered_files = [
            f for f in theraflow_files 
            if any(p.lower() in f.lower() or p.lower().replace("docking_", "run_docking_") in f.lower() for p in target_prefixes)
        ]
        if not filtered_files:
            print("ℹ️ Target prefixes did not match specific TheraFlow runs. Displaying all available TheraFlow docking results.")
            filtered_files = theraflow_files
    else:
        filtered_files = theraflow_files
        
    cif_files = sorted(filtered_files)
    # Human-readable labels: e.g. "run_docking_bicev_hcc_11_model (01_prepare/complex_clean.pdb)"
    cif_options = {
        f"{os.path.basename(os.path.dirname(os.path.dirname(f)))} ({os.path.basename(os.path.dirname(f))}/{os.path.basename(f)})": f 
        for f in cif_files
    }
    print(f"✅ [TheraFlow] Loaded {len(cif_options)} docking/MD structure(s).")

elif af3_files:
    # Filter by target_prefixes if defined
    if target_prefixes:
        filtered_files = [
            f for f in af3_files 
            if any(os.path.basename(f).startswith(prefix) for prefix in target_prefixes)
        ]
        # Fall back to all AF3 files if no prefixes matched
        if not filtered_files:
            print("⚠️ None of the target prefixes matched AlphaFold 3 files. Showing all AF3 models.")
            filtered_files = af3_files
    else:
        filtered_files = af3_files
        
    cif_files = sorted(filtered_files)
    cif_options = {os.path.basename(f).replace("_model.cif", ""): f for f in cif_files}
    print(f"✅ [AlphaFold 3] Loaded {len(cif_options)} model(s).")

else:
    print("❌ No docking results found in 'TheraFlow/outputs/docking_md_runs' or 'alphafold3/output_result5'.")
    cif_options = {}

# ==============================================================================
# 2. Interactive Widget Dashboard & Snapshot Controls
# ==============================================================================

model_select = widgets.Dropdown(
    options=cif_options,
    description='Model:',
    layout=widgets.Layout(width='480px')
)

mode_select = widgets.Dropdown(
    options={'Standard Cartoon': 'cartoon', 'Lock & Key Surface': 'lock_and_key', 'Super Clean': 'super_clean'},
    value='cartoon',
    description='Style:',
    layout=widgets.Layout(width='280px')
)

output_area = widgets.Output()

def update_view(change=None):
    with output_area:
        clear_output(wait=True)
        if not model_select.value:
            print("No model selected.")
            return
        cif_path = model_select.value
        view_mode = mode_select.value
        ligand_resn = detect_ligand_resn(cif_path)
        print(f"Target File: {cif_path}")
        print(f"Detected Ligand Code: {ligand_resn}")
        if view_mode == 'lock_and_key':
            view = view_lock_and_key(cif_path, ligand_resn)
        elif view_mode == 'super_clean':
            view = view_super_clean(cif_path, ligand_resn)
        else:
            view = view_docked_pocket(cif_path, ligand_resn)
        view.show()

model_select.observe(update_view, names='value')
mode_select.observe(update_view, names='value')

btn = widgets.Button(
    description="Save Current View (PNG)",
    button_style="success",
    icon="camera",
    layout=widgets.Layout(width='250px')
)

def capture_snapshot(b):
    js = """
    var canvases = document.querySelectorAll('canvas');
    if (canvases.length > 0) {
        var canvas = canvases[canvases.length - 1];
        var img = canvas.toDataURL('image/png');
        var link = document.createElement('a');
        link.download = 'binding_pocket_snapshot.png';
        link.href = img;
        link.click();
    } else {
        alert('Could not find WebGL canvas in this cell output!');
    }
    """
    display(Javascript(js))

btn.on_click(capture_snapshot)

controls = widgets.VBox([
    widgets.HBox([model_select, mode_select]),
    widgets.Box([btn], layout=widgets.Layout(margin='10px 0px 10px 120px'))
])

display(controls, output_area)
update_view()


✅ [AlphaFold 3] Loaded 4 model(s).


Output()